## T4 GPU benchmark — Experiment N model set

Local runs are too slow to run 10+ full experiments at N=1500 (measured ~24.7 hr per run, see
`notebooks/03_cuad_rag_scale.ipynb`). Before committing budget/time to a paid batch, this notebook
benchmarks a single infra variable — an Nvidia T4 GPU (HF Space `Nvidia T4 - small`, $0.40/hr) —
while holding the model set identical to Experiment N (the current best config):

- Embedder: `nomic-embed-text-v2-moe`
- Generator: `mistral-small3.2:24b`
- Judge / HyDE: `llama3.1:8b-instruct-q4_K_M`

Keeping the models fixed isolates "how much does T4 speed things up" from "which model should we
use" — the model set is free to change once we know what infra to run the real batch on (see
`evaluation/infra_scaling_trials.md`).

`mistral-small3.2:24b` at q4 quantization is a tight fit in T4's 16GB VRAM — watch for OOM as well
as raw speed. N is kept small (20) since this is purely a timing probe, not a quality measurement.

In [ ]:
import os
import sys
import subprocess
import time

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text-v2-moe:latest
    !ollama pull llama3.1:8b-instruct-q4_K_M
    !ollama pull mistral-small3.2:24b

# On an HF Space with GPU hardware: install Ollama the same way inside the Space's Docker image
# and pull the same three models — PROVIDER stays "ollama", only the underlying hardware changes.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import PROMPT_V2, build_rag_chain
from reliablerag.env import load_secrets
from reliablerag.experiment import evaluate_results, run_rag_experiment
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import get_hyde_retriever, get_or_build_vector_store, get_reranker, get_retriever

### 1. Configuration

In [ ]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
JUDGE_MODEL        = os.environ["JUDGE_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]
GENERATOR_MODEL    = "mistral-small3.2:24b"  # pinned to Exp N regardless of .env override

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"Judge model     : {JUDGE_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

In [ ]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)
llm        = create_llm(PROVIDER, GENERATOR_MODEL)
judge_llm  = create_llm(PROVIDER, JUDGE_MODEL, temperature=0)
hyde_llm   = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")

### 2. Load a small CUAD slice (N=20 — timing probe, not a quality measurement)

In [ ]:
N_SAMPLES = 20

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))
print(f"Loaded {len(samples)} CUAD samples")

### 3. Run Experiment N config — timed

In [ ]:
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_t4bench"

t0 = time.perf_counter()
results_t4 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=llm,
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)
gen_elapsed = time.perf_counter() - t0
print(f"\n[BENCHMARK] Generation phase — {gen_elapsed:.1f}s total, {gen_elapsed/len(samples):.2f}s/sample")

In [ ]:
t0 = time.perf_counter()
agg_t4 = evaluate_results(results_t4, judge_llm, n_runs=1)
judge_elapsed = time.perf_counter() - t0
print(f"\n[BENCHMARK] Judge phase — {judge_elapsed:.1f}s total, {judge_elapsed/len(results_t4):.2f}s/sample")
print(f"Exp N on T4 (N={len(results_t4)}) — Rel {agg_t4['avg_relevance']:.3f} / Util {agg_t4['avg_utilization']:.3f} / Comp {agg_t4['avg_completeness']:.3f} / Adh {agg_t4['adherence_rate']:.0%}")

### 4. Compare to local baseline

Local baseline (measured in `notebooks/03_cuad_rag_scale.ipynb`, N=100): generation 39.2s/sample,
judge 19.0s/sample. Record the T4 numbers below in `evaluation/infra_scaling_trials.md`.

In [ ]:
LOCAL_GEN_S_PER_SAMPLE   = 39.2
LOCAL_JUDGE_S_PER_SAMPLE = 19.0

t4_gen_per_sample   = gen_elapsed / len(samples)
t4_judge_per_sample = judge_elapsed / len(results_t4)

print(f"Generation: local {LOCAL_GEN_S_PER_SAMPLE:.2f}s/sample -> T4 {t4_gen_per_sample:.2f}s/sample "
      f"({LOCAL_GEN_S_PER_SAMPLE / t4_gen_per_sample:.2f}x)")
print(f"Judge     : local {LOCAL_JUDGE_S_PER_SAMPLE:.2f}s/sample -> T4 {t4_judge_per_sample:.2f}s/sample "
      f"({LOCAL_JUDGE_S_PER_SAMPLE / t4_judge_per_sample:.2f}x)")